# AML Detection - XGBoost / LightGBM / Random Forest Ensemble

Same data cleaning, transaction-graph construction, and feature engineering as the
GNN version (`data_processing.py`, `graph_construction.py`, `feature_engineering.py`
are unchanged) -- the graph is still built, but only to *compute features*
(predecessor/successor counts, fan-in/out, relay timing, short-cycle detection),
not to feed a graph neural network. Everything downstream of the engineered
feature table is now a tabular tree ensemble: no PyG `Data` object, no custom
neighbor sampler, no epochs.


In [ ]:
import gc
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from data_processing import load_and_clean, vocab_sizes
from graph_construction import build_transaction_graph
from feature_engineering import engineer_all_features
from feature_columns import NUMERIC_FEATURE_COLUMNS, CATEGORICAL_FEATURE_COLUMNS
from model import AMLEnsemble
from train import (
    train_pipeline,
    run_inference,
    compute_classification_metrics,
    print_metrics_report,
)
from aggregation import build_transaction_view, aggregate_accounts

np.random.seed(0)

# ==========================================================
# Cache directory
# ==========================================================

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

VOCABS_PATH = CACHE_DIR / "vocabs.pkl"

TRAIN_ENGINEERED_PATH = CACHE_DIR / "df_train_engineered.parquet"
TEST_ENGINEERED_PATH = CACHE_DIR / "df_test_engineered.parquet"

ENSEMBLE_PATH = CACHE_DIR / "ensemble.joblib"

In [ ]:
TRAIN_CSV = "HI-Small_Trans.csv"
TEST_CSV = "LI-Small_Trans.csv"

pd.read_csv(TRAIN_CSV, nrows=5)

In [ ]:
df_train, vocabs = load_and_clean(TRAIN_CSV)

if VOCABS_PATH.exists():
    print("Loading cached vocabulary...")
    with open(VOCABS_PATH, "rb") as f:
        vocabs = pickle.load(f)
else:
    print("Saving vocabulary...")
    with open(VOCABS_PATH, "wb") as f:
        pickle.dump(vocabs, f)

print(f"Train: {len(df_train):,} transactions, laundering rate {df_train['label'].mean():.3%}")
print(f"Memory (deep): {df_train.memory_usage(deep=True).sum() / 1e9:.3f} GB")

vocab_sizes(vocabs)

## Transaction graph (feature engineering only)

Still built with the same vectorized NetworKit construction as the GNN version --
the graph itself never touches the model now, it only feeds `feature_engineering.py`
(predecessor/successor CSR adjacency -> fan-in/out, relay timing, short-cycle flags).

In [ ]:
g_train, src_train, dst_train, preds_train, succs_train = build_transaction_graph(df_train)

print(
    f"Train graph: {g_train.numberOfNodes():,} nodes, "
    f"{g_train.numberOfEdges():,} edges "
    f"(avg out-degree {g_train.numberOfEdges()/g_train.numberOfNodes():.2f})"
)

In [ ]:
if TRAIN_ENGINEERED_PATH.exists():
    print("Loading cached engineered training features...")
    df_train = pd.read_parquet(TRAIN_ENGINEERED_PATH)
else:
    print("Engineering training features...")
    df_train = engineer_all_features(
        df_train, preds_train, succs_train, src_train, dst_train,
    )
    df_train.to_parquet(TRAIN_ENGINEERED_PATH, index=False)
    print("Saved engineered training features.")

print(
    f"Engineered feature count: "
    f"{len(NUMERIC_FEATURE_COLUMNS)} numeric + "
    f"{len(CATEGORICAL_FEATURE_COLUMNS)} categorical"
)

df_train.filter(
    regex="^(sender_|receiver_|pair_|fan_|relay_|short_cycle)"
).head()

## Train the ensemble

`train_pipeline` does the chronological train/val split, fits XGBoost + LightGBM
(each with internal early stopping against the val split) + Random Forest, fits the
logistic-regression stacker on val-set base predictions, and picks the F1-optimal
decision threshold on val -- all in one call, all CPU, no GPU required.

In [ ]:
ensemble, optimal_threshold, df_val, val_probs = train_pipeline(
    df_train,
    val_frac=0.15,
    neg_per_pos=None,   # set e.g. 20.0 to subsample negatives on very large files
    checkpoint_path=ENSEMBLE_PATH,
)

val_metrics = compute_classification_metrics(
    df_val["label"].values, val_probs, threshold=optimal_threshold,
)
print_metrics_report(val_metrics, title="VALIDATION SET METRICS")

In [ ]:
fig = px.histogram(
    x=val_probs, nbins=60, title="Validation-set risk score distribution",
    labels={"x": "Ensemble risk score"},
)
fig.add_vline(x=optimal_threshold, line_dash="dash", line_color="red")
fig.show()

In [ ]:
feature_importance = ensemble.feature_importance(top_n=25)
feature_importance.to_csv("feature_importance.csv", index=False)

fig = px.bar(
    feature_importance.sort_values("average"),
    x="average", y="feature", orientation="h",
    title="Top 25 features - average gain across XGBoost / LightGBM / Random Forest",
)
fig.show()

In [ ]:
print("Saved ensemble to", ENSEMBLE_PATH)

del g_train, src_train, dst_train, preds_train, succs_train, df_train
gc.collect()
print("Freed training graph and feature objects.")

## Test set: clean -> graph features -> score

The categorical vocabulary fit on train is reused here (`vocabs`), so
test-only accounts/banks fall into the shared UNK bucket instead of
crashing -- unchanged from the GNN version, since this is a
`data_processing.py` concern, not a model concern.

In [ ]:
df_test, _ = load_and_clean(TEST_CSV, vocabs=vocabs)

print(f"Test: {len(df_test):,} transactions, laundering rate {df_test['label'].mean():.3%}")
print(f"Memory (deep): {df_test.memory_usage(deep=True).sum()/1e9:.3f} GB")

In [ ]:
g_test, src_test, dst_test, preds_test, succs_test = build_transaction_graph(df_test)

print(f"Test graph: {g_test.numberOfNodes():,} nodes, {g_test.numberOfEdges():,} edges")

if TEST_ENGINEERED_PATH.exists():
    print("Loading cached engineered test features...")
    df_test = pd.read_parquet(TEST_ENGINEERED_PATH)
else:
    print("Engineering test features...")
    df_test = engineer_all_features(df_test, preds_test, succs_test, src_test, dst_test)
    df_test.to_parquet(TEST_ENGINEERED_PATH, index=False)
    print("Saved engineered test features.")

In [ ]:
test_probs = run_inference(ensemble, df_test)
test_y = df_test["label"].values

metrics = compute_classification_metrics(test_y, test_probs, threshold=optimal_threshold)
print_metrics_report(metrics)

cm = metrics["confusion_matrix"]
fig = px.imshow(
    cm, text_auto=True, color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=["Legitimate (0)", "Laundering (1)"], y=["Legitimate (0)", "Laundering (1)"],
    title="Confusion matrix - test set",
)
fig.show()

## Reload check

Sanity check that the saved ensemble reproduces identical probabilities after a
fresh load -- the tabular equivalent of the GNN notebook's `state_dict` reload cell.

In [ ]:
reloaded = AMLEnsemble.load(ENSEMBLE_PATH)
probs_reloaded = run_inference(reloaded, df_test)
print("Max abs difference vs. original ensemble's probabilities:", np.abs(test_probs - probs_reloaded).max())

## Export for the dashboard (PRD Section 14-16)

In [ ]:
RISK_THRESHOLD = optimal_threshold

transaction_view = build_transaction_view(df_test, test_probs, threshold=RISK_THRESHOLD)
account_view = aggregate_accounts(df_test, test_probs, threshold=RISK_THRESHOLD)

transaction_view.to_csv("transaction_view.csv", index=False)
account_view.to_csv("account_view.csv", index=False)

print(f"{transaction_view['Flagged'].sum():,} / {len(transaction_view):,} transactions flagged "
      f"at threshold {RISK_THRESHOLD:.4f}")
print(f"{account_view['Account Alert'].sum():,} / {len(account_view):,} accounts alerted")
transaction_view.head(10)

In [ ]:
account_view.head(10)